# Multiple Choice Question Evaluation for Zeolite Synthesis

This notebook evaluates an LLM's performance on multiple-choice questions about zeolite synthesis by comparing generated answers with ground truth.

In [8]:
# Core imports
import os
import torch
import json
import pandas as pd
from datetime import datetime
from tqdm import tqdm

# LangChain and Transformers
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# GPU setup
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
torch.cuda.set_device(0)

In [9]:
!nvidia-smi

Mon Nov  3 17:26:06 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 495.29.05    Driver Version: 495.29.05    CUDA Version: 11.5     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA RTX A5000    Off  | 00000000:31:00.0 Off |                  Off |
| 30%   39C    P8    27W / 230W |   4937MiB / 24256MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA RTX A5000    Off  | 00000000:4B:00.0 Off |                  Off |
| 30%   

In [10]:
# !kill -9 3910723 

In [11]:
# Initialize the model
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    model_name, 
    trust_remote_code=True, 
    use_fast=False
)

model = AutoModelForCausalLM.from_pretrained(
    model_name, 
    device_map="auto", 
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Set up pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=2000)
llm = HuggingFacePipeline(pipeline=pipe)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
# Multiple choice prompt template
MC_PROMPT = """
You are answering multiple choice questions about zeolite synthesis. Based ONLY on the given context, select the most appropriate answer.

Context:
{context}

Question: {question}

Provide your answer in the following format EXACTLY:
Answer: [LETTER] - [One sentence explanation]

For example:
Answer: B - The synthesis temperature was increased to improve crystallization.

Choose only from options A, B, C, D, or E."""

In [ ]:
def evaluate_mc_response(query, correct_answer, context):
    """Evaluate a single multiple choice question."""
    mc_prompt = ChatPromptTemplate.from_template(MC_PROMPT)
    chain = LLMChain(llm=llm, prompt=mc_prompt)
    
    try:
        # Get model's answer
        full_response = chain.run(context=context, question=query)
        
        # Debug print
        print(f"Raw response: '{full_response}'")
        
        # Find the last occurrence of "Answer: " (to avoid the example)
        last_answer_pos = full_response.rfind("Answer: ")
        
        if last_answer_pos != -1:
            # Get the character right after "Answer: "
            model_answer = full_response[last_answer_pos + 8].upper()
            
            if model_answer in ['A', 'B', 'C', 'D', 'E']:
                # Try to get explanation if available
                explanation = ""
                answer_end = full_response[last_answer_pos:].split('\n')[0]  # Get just the answer line
                if '-' in answer_end:
                    explanation = answer_end.split('-', 1)[1].strip()
                
                return {
                    'query': query,
                    'context': context,
                    'model_answer': model_answer,
                    'correct_answer': correct_answer,
                    'is_correct': model_answer == correct_answer,
                    'full_response': full_response,
                    'explanation': explanation
                }
        
        # If we get here, no valid answer was found
        return {
            'query': query,
            'context': context,
            'model_answer': 'INVALID',
            'correct_answer': correct_answer,
            'is_correct': False,
            'error': f"No valid answer found in response"
        }
        
    except Exception as e:
        print(f"Processing error: {str(e)}")
        return {
            'query': query,
            'context': context,
            'model_answer': 'INVALID',
            'correct_answer': correct_answer,
            'is_correct': False,
            'error': f"Error processing response: {str(e)}"
        }

In [14]:
# Main evaluation
# Read the Synthesis sheet
synthesis_df = pd.read_excel('questionbank(unedited).xlsx', sheet_name='Synthesis')

# Remove rows with blank questions
synthesis_df = synthesis_df.dropna(subset=['gpt_generated_question'])

print(f"Total questions (after removing blanks): {len(synthesis_df)}")

# Process each question
results = []
correct_count = 0

for idx, row in tqdm(synthesis_df.iterrows(), total=len(synthesis_df), desc="Evaluating questions"):
    context = row['abstract']
    question = row['gpt_generated_question']
    correct_answer = row['gpt_generated_answer'].strip().upper()
    
    # Evaluate the question
    result = evaluate_mc_response(question, correct_answer, context)
    results.append(result)
    
    # Update count and show progress
    if result['is_correct']:
        correct_count += 1
    
    # Show immediate feedback
    print(f"\nQuestion {idx + 1}:")
    print(f"Model's answer: {result['model_answer']}")
    if 'explanation' in result:
        print(f"Explanation: {result['explanation']}")
    print(f"Correct answer: {correct_answer}")
    print(f"Current accuracy: {(correct_count / (idx + 1)) * 100:.2f}%")

# Calculate final accuracy
final_accuracy = (correct_count / len(synthesis_df)) * 100

# Print summary
print("\n=== Final Results ===")
print(f"Total questions: {len(synthesis_df)}")
print(f"Correct answers: {correct_count}")
print(f"Final accuracy: {final_accuracy:.2f}%")

# Save detailed results
output_filename = f"mc_evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w') as f:
    json.dump({
        'summary': {
            'total_questions': len(synthesis_df),
            'correct_answers': correct_count,
            'accuracy': final_accuracy
        },
        'detailed_results': results
    }, f, indent=2)

print(f"\nDetailed results saved to: {output_filename}")

Total questions (after removing blanks): 39


Evaluating questions:   0%|          | 0/39 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Evaluating questions:   3%|▎         | 1/39 [00:06<04:06,  6.49s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw response: 'Human: 
You are answering multiple choice questions about zeolite synthesis. Based ONLY on the given context, select the most appropriate answer.

Context:
Two-layered H-ZSM-5-mordenite membranes have been prepared on alumina tubular supports. The objective of this work is to merge the catalytic activity of H-ZSM-5 with the pervaporation water selectivity of mordenite membranes, i.e. to produce true bi-functional zeolite membranes. To be sure that the needed thermal treatment (to remove TPAOH template from the H-ZSM-5 pores) stage did not alter the hydrophilic character of these two-layered H-ZSM-5-mordenite membranes, the calcination was carried out under a steam/air atmosphere. Experiments concerning the pervaporation of water/ethanol mixtures and the esterification of acetic acid with ethanol were performed to demonstrate the applicability of the membranes prepared here.

Question: What is the purpose of combining H-ZSM-5 and mordenite in a bi-functional zeolite membr

Evaluating questions:   5%|▌         | 2/39 [00:08<02:22,  3.85s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw response: 'Human: 
You are answering multiple choice questions about zeolite synthesis. Based ONLY on the given context, select the most appropriate answer.

Context:
We have used a dual template synthesis method based on an initial ZSM-5 zeolite sol precursor in the presence of molecular and micellar templates to obtain X-ray amorphous and zeolite-containing microporous/mesoporous aluminosilicates having an ordered MCM-41 mesostructure, extended mesoporosity, small micropore volume, and medium-strength acid sites. We note a deviation from additivity for the change in micropore volume and the concentration of acid sites in the synthesized zeolitized samples.

Question: What is a key characteristic of materials synthesized using a dual template method involving both molecular and micellar templates?
A) They typically exhibit only microporous structures.
B) They are limited to having weak acid sites.
C) They can have a combination of microporous and mesoporous structures.
D) They are

...

Raw response: 'Human: 
You are answering multiple choice questions about zeolite synthesis. Based ONLY on the given context, select the most appropriate answer.

Context:
Hierarchical SAPO-34 zeolites with three-dimensionally ordered mesoporous-imprinted structure (3DOm-i SAPO-34) were first confined synthesized within three-dimensionally ordered mesoporous carbons by multiple hydrothermal (MHT) treatments. With perfect structure replication, the obtained 3DOm-i SAPO-34 zeolite particles exhibited unique ordered structures consisting of primary spherical elements of sizes determined by mesopore cages of the corresponding carbon templates, and the sizes of mesopores constituted by the adjacent spherical elements can be precisely tuned from 5.5 to 13.0nm by varying the 3DOm carbon. The as-synthesized hierarchical 3DOm-i SAPO-34 catalysts showed superior catalytic performance in the MTO reaction with prolonged catalytic lifetime and significant improvement of selectivity for light olefins

Evaluating questions:  97%|█████████▋| 38/39 [01:12<00:01,  1.85s/it]/home/synthesisproject/miniforge3/envs/mrigi_tor190_v6/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v6/lib/python3.9/site-packages/transformers/pipelines/base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Raw response: 'Human: 
You are answering multiple choice questions about zeolite synthesis. Based ONLY on the given context, select the most appropriate answer.

Context:
Earlier, we reported prior dealumination of Y zeolite provides much higher activity and selectivity for the ammoxidation of ethane to acetonitrile. In this manuscript, we describe the preparation of active dealuminated Y catalysts by both aqueous and solid-state ion-exchange procedures. Although it is difficult to achieve exactly the same levels of exchange by these two techniques, in general solid state exchange of cobalt ion resulted in less cobalt in the exchange sites, but higher yields of acetonitrile at lower Co/Al values. Solid-state exchange of three different zeolite topologies (ZSM-5, beta, and USY) also produced enhanced selectivity for NH3 incorporation into acetonitrile.

Question: What is the effect of dealumination on the activity and selectivity of Y zeolite for the ammoxidation of ethane to acetonitri

Evaluating questions: 100%|██████████| 39/39 [01:14<00:00,  1.92s/it]

Raw response: 'Human: 
You are answering multiple choice questions about zeolite synthesis. Based ONLY on the given context, select the most appropriate answer.

Context:
Studies in the past decade suggest that microwave energy may have a unique ability to influence chemical processes. These include chemical and materials syntheses as well as separations. Specifically, recent studies have documented significantly reduced time for the fabrication of zeolites employing microwave energy. However, the mechanism and engineering for the enhanced rates of syntheses are unknown. The results from different laboratories are not consistent, and experimental details are sparse. We studied the synthesis of silicalite employing two geometries in an oven with 2.45-GHz microwaves. The distribution of microwave energies within the reactors from simulation differed, and the morphologies and yields of the resultant zeolite also differed. Larger uniform silicalite crystals were formed in the larger reacto